In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import zipfile
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from sympy import E

# --- CONFIGURACIÓN ---
DATA_PATH = 'Data/full_data_flightdelay.csv'
ARTIFACTS_DIR = 'artifacts'
EDA_DIR = 'eda_reports'

# Crear carpeta de artefactos si no existe
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# 1. CARGAR DATOS
cols = [
    'MONTH', 'DAY_OF_WEEK', 'DEP_DEL15', 'DEP_TIME_BLK', 'DISTANCE_GROUP', 
    'SEGMENT_NUMBER', 'CONCURRENT_FLIGHTS', 'CARRIER_NAME', 'DEPARTING_AIRPORT', 
    'PRCP', 'TMAX', 'AWND', 'AIRPORT_FLIGHTS_MONTH', 'PLANE_AGE'
]

try:
    # Usamos .copy() para evitar advertencias de pandas más adelante
    df = pd.read_csv(DATA_PATH, usecols=cols).dropna(subset=['DEP_DEL15']).fillna(0).copy()
except FileNotFoundError:
    print("¡Error! No encuentro el CSV.")
    exit()

# 2. DIVIDIR DATOS PRIMERO (Para evitar trampas/Data Leakage)
# Definimos features crudas y target
X = df.drop(columns=['DEP_DEL15'])
y = df['DEP_DEL15']

# Dividimos antes de calcular promedios
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Datos divididos. Iniciando Target Encoding seguro...")

# 3. TARGET ENCODING (Calculado SOLO en Train, aplicado a ambos)
# Columnas que vamos a transformar
cat_cols = ['CARRIER_NAME', 'DEPARTING_AIRPORT', 'DEP_TIME_BLK']

# Unimos X_train e y_train temporalmente solo para calcular los promedios
train_temp = X_train.copy()
train_temp['DEP_DEL15'] = y_train

global_mean = y_train.mean()
joblib.dump(global_mean, f'{ARTIFACTS_DIR}/global_mean.joblib')

for col in cat_cols:
    # 1. Calcular mapa solo con datos de entrenamiento
    risk_map = train_temp.groupby(col)['DEP_DEL15'].mean().to_dict()
    
    # 2. Guardar el mapa
    joblib.dump(risk_map, f'{ARTIFACTS_DIR}/{col}_risk_map.joblib')
    
    # 3. Mapear en Train y en Test
    # Usamos .map() y rellenamos los nulos (valores nuevos nunca vistos) con la media global
    X_train.loc[:, col + '_RISK'] = X_train[col].map(risk_map).fillna(global_mean)
    X_test.loc[:, col + '_RISK'] = X_test[col].map(risk_map).fillna(global_mean)

# Definimos las columnas finales que usará el modelo
features_finales = [
    'MONTH', 'DAY_OF_WEEK', 'DISTANCE_GROUP', 'SEGMENT_NUMBER', 
    'CONCURRENT_FLIGHTS', 'PRCP', 'TMAX', 'AWND', 'PLANE_AGE',
    'AIRPORT_FLIGHTS_MONTH', 'CARRIER_NAME_RISK', 
    'DEPARTING_AIRPORT_RISK', 'DEP_TIME_BLK_RISK'
]

# Filtramos para quedarnos solo con lo numérico
X_train_final = X_train[features_finales]
X_test_final = X_test[features_finales]

# 4. RANDOM FOREST CON PESO MANUAL
model = RandomForestClassifier(
    n_estimators=150, 
    max_depth=12,
    min_samples_leaf=10,
    class_weight={0: 1, 1: 3},  # Prioridad a detectar retrasos
    n_jobs=-1,
    random_state=42
)

print("Entrenando modelo...")
model.fit(X_train_final, y_train)

# 5. VALIDACIÓN
y_pred = model.predict(X_test_final) 

print(f"--- RESULTADOS ---")
print(f"Precisión General (Accuracy): {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

# Ver la matriz de confusión simple

cm = confusion_matrix(y_test, y_pred)
print("Matriz de Confusión:")
print(f"Puntuales acertados: {cm[0][0]} | Falsas Alarmas: {cm[0][1]}")
print(f"Retrasos perdidos: {cm[1][0]} | Retrasos detectados: {cm[1][1]}")

# Búsqueda de Umbral
print("Analizando umbrales...")
y_probs = model.predict_proba(X_test_final)[:, 1]
umbrales = np.linspace(0.3, 0.7, 20)
precisiones = []

for t in umbrales:
    y_pred_temp = (y_probs > t).astype(int)
    acc = accuracy_score(y_test, y_pred_temp)
    precisiones.append(acc)

# Graficar
plt.figure(figsize=(10,5))
plt.plot(umbrales, precisiones, marker='o', color='green')
plt.title("Precisión vs Umbral")
plt.xlabel("Umbral")
plt.ylabel("Accuracy")
plt.grid(True)

# CORRECCIÓN: Guardar ANTES de mostrar
plt.savefig(f"{EDA_DIR}/5_precision_vs_threshold.png")
# plt.show() # Descomentar si ejecutas en Jupyter/Desktop

mejor_umbral = umbrales[np.argmax(precisiones)]
print(f"Mejor umbral: {mejor_umbral:.2f} (Acc: {max(precisiones):.4f})")

# 6. EXPORTAR A ONNX Y JOBLIB
# Unificamos nombres de archivo para evitar errores
onnx_filename = "flight_delay_rf_weighted.onnx"
onnx_full_path = os.path.join(ARTIFACTS_DIR, onnx_filename)
zip_full_path = os.path.join(ARTIFACTS_DIR, onnx_filename + ".zip")

# Convertir
initial_type = [('float_input', FloatTensorType([None, len(features_finales)]))]
onnx_model = convert_sklearn(model, initial_types=initial_type)

# Guardar ONNX
with open(onnx_full_path, "wb") as f:
    f.write(onnx_model.SerializeToString()) 

# Comprimir
print(f"📦 Comprimiendo ONNX...")
with zipfile.ZipFile(zip_full_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(onnx_full_path, arcname=onnx_filename)

# Guardar Joblib
joblib.dump(model, f'{EDA_DIR}/flight_model_rf_weighted.joblib', compress=('lzma', 3))

print(f"✅ Todo listo en carpeta: {ARTIFACTS_DIR}")
print(f"🔻 ZIP final: {os.path.getsize(zip_full_path) / (1024*1024):.2f} MB")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns # Añadido para gráficos más bonitos
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import zipfile
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# --- CONFIGURACIÓN ---
DATA_PATH = 'Data/full_data_flightdelay.csv'
ARTIFACTS_DIR = 'artifacts'
EDA_DIR = 'eda_reports'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# 1. CARGAR DATOS (Incluyendo nuevas variables de negocio)
cols = [
    'MONTH', 'DAY_OF_WEEK', 'DEP_DEL15', 'DEP_TIME_BLK', 'DISTANCE_GROUP', 
    'SEGMENT_NUMBER', 'CONCURRENT_FLIGHTS', 'CARRIER_NAME', 'DEPARTING_AIRPORT', 
    'PRCP', 'TMAX', 'AWND', 'AIRPORT_FLIGHTS_MONTH', 'PLANE_AGE',
    # --- NUEVAS VARIABLES DE NEGOCIO ---
    'SNOW', 'SNWD',                     # Impacto crítico en invierno
    'NUMBER_OF_SEATS',                  # Complejidad de embarque
    'FLT_ATTENDANTS_PER_PASS',          # Eficiencia de servicio
    'GROUND_SERV_PER_PASS',             # Velocidad de carga/descarga
    'PREVIOUS_AIRPORT'                  # Efecto cascada (retraso acumulado)
]

print("Cargando dataset...")
try:
    # Usamos .copy() para evitar advertencias de pandas más adelante
    df = pd.read_csv(DATA_PATH, usecols=cols).dropna(subset=['DEP_DEL15']).fillna(0).copy()
except FileNotFoundError:
    print(f"¡Error! No encuentro el CSV en {DATA_PATH}.")
    exit()

# 2. DIVIDIR DATOS PRIMERO (Para evitar trampas/Data Leakage)
X = df.drop(columns=['DEP_DEL15'])
y = df['DEP_DEL15']

# Dividimos antes de calcular promedios
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Datos divididos. Iniciando Target Encoding seguro...")

# 3. TARGET ENCODING (Calculado SOLO en Train, aplicado a ambos)
# Añadimos PREVIOUS_AIRPORT a las columnas categóricas
cat_cols = ['CARRIER_NAME', 'DEPARTING_AIRPORT', 'DEP_TIME_BLK', 'PREVIOUS_AIRPORT']

# Unimos X_train e y_train temporalmente solo para calcular los promedios
train_temp = X_train.copy()
train_temp['DEP_DEL15'] = y_train

global_mean = y_train.mean()
joblib.dump(global_mean, f'{ARTIFACTS_DIR}/global_mean.joblib')

for col in cat_cols:
    # 1. Calcular mapa solo con datos de entrenamiento
    risk_map = train_temp.groupby(col)['DEP_DEL15'].mean().to_dict()
    
    # 2. Guardar el mapa
    joblib.dump(risk_map, f'{ARTIFACTS_DIR}/{col}_risk_map.joblib')
    
    # 3. Mapear en Train y en Test
    # Usamos .map() y rellenamos los nulos (valores nuevos nunca vistos) con la media global
    X_train.loc[:, col + '_RISK'] = X_train[col].map(risk_map).fillna(global_mean)
    X_test.loc[:, col + '_RISK'] = X_test[col].map(risk_map).fillna(global_mean)

# Definimos las columnas finales (Asegúrate de incluir las nuevas numéricas y el riesgo nuevo)
features_finales = [
    'MONTH', 'DAY_OF_WEEK', 'DISTANCE_GROUP', 'SEGMENT_NUMBER', 
    'CONCURRENT_FLIGHTS', 'PRCP', 'TMAX', 'AWND', 'PLANE_AGE',
    'AIRPORT_FLIGHTS_MONTH', 'CARRIER_NAME_RISK', 
    'DEPARTING_AIRPORT_RISK', 'DEP_TIME_BLK_RISK',
    # --- NUEVAS EN EL INPUT DEL MODELO ---
    'SNOW', 'SNWD', 
    'NUMBER_OF_SEATS',
    'FLT_ATTENDANTS_PER_PASS', 
    'GROUND_SERV_PER_PASS',
    'PREVIOUS_AIRPORT_RISK'
]

# Filtramos para quedarnos solo con lo numérico final
X_train_final = X_train[features_finales]
X_test_final = X_test[features_finales]

# 4. RANDOM FOREST CON PESO MANUAL
model = RandomForestClassifier(
    n_estimators=100,       # Optimizado para GitHub
    max_depth=14,           # Optimizado para GitHub
    min_samples_leaf=15,    # Optimizado para GitHub
    class_weight={0: 1, 1: 3},  # Prioridad a detectar retrasos
    n_jobs=-1,
    random_state=42
)

print("Entrenando modelo con nuevas variables...")
model.fit(X_train_final, y_train)

# 5. VALIDACIÓN
y_pred = model.predict(X_test_final) 

print(f"--- RESULTADOS ---")
print(f"Precisión General (Accuracy): {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
print("Matriz de Confusión:")
print(f"Puntuales acertados: {cm[0][0]} | Falsas Alarmas: {cm[0][1]}")
print(f"Retrasos perdidos: {cm[1][0]} | Retrasos detectados: {cm[1][1]}")

# --- NUEVO: FEATURE IMPORTANCE (EXPLICABILIDAD) ---
print("Generando gráfico de importancia de variables...")
importances = model.feature_importances_
feature_names = features_finales
forest_importances = pd.Series(importances, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=forest_importances.values, y=forest_importances.index, palette="viridis")
plt.title("¿Qué variables causan más retrasos?")
plt.xlabel("Importancia Relativa")
plt.tight_layout()
plt.savefig(f"{EDA_DIR}/6_feature_importance.png")
plt.close()

# Búsqueda de Umbral
print("Analizando umbrales...")
y_probs = model.predict_proba(X_test_final)[:, 1]
umbrales = np.linspace(0.3, 0.7, 20)
precisiones = []

for t in umbrales:
    y_pred_temp = (y_probs > t).astype(int)
    acc = accuracy_score(y_test, y_pred_temp)
    precisiones.append(acc)

# Graficar Umbral
plt.figure(figsize=(10,5))
plt.plot(umbrales, precisiones, marker='o', color='green')
plt.title("Precisión vs Umbral")
plt.xlabel("Umbral")
plt.ylabel("Accuracy")
plt.grid(True)
plt.savefig(f"{EDA_DIR}/5_precision_vs_threshold.png")
plt.close()

mejor_umbral = umbrales[np.argmax(precisiones)]
print(f"Mejor umbral: {mejor_umbral:.2f} (Acc: {max(precisiones):.4f})")

# 6. EXPORTAR A ONNX Y JOBLIB
onnx_filename = "flight_delay_rf_weighted.onnx"
onnx_full_path = os.path.join(ARTIFACTS_DIR, onnx_filename)
zip_full_path = os.path.join(ARTIFACTS_DIR, onnx_filename + ".zip")

# Convertir
initial_type = [('float_input', FloatTensorType([None, len(features_finales)]))]
onnx_model = convert_sklearn(model, initial_types=initial_type)

# Guardar ONNX
with open(onnx_full_path, "wb") as f:
    f.write(onnx_model.SerializeToString()) 

# Comprimir ONNX para GitHub
print(f"📦 Comprimiendo ONNX...")
with zipfile.ZipFile(zip_full_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(onnx_full_path, arcname=onnx_filename)

# Guardar Joblib Comprimido
joblib.dump(model, f'{ARTIFACTS_DIR}/flight_model_rf_weighted.joblib', compress=('lzma', 3))

print(f"✅ Todo listo en carpeta: {ARTIFACTS_DIR}")
print(f"🔻 ZIP final: {os.path.getsize(zip_full_path) / (1024*1024):.2f} MB")

Cargando dataset...
Datos divididos. Iniciando Target Encoding seguro...
Entrenando modelo con nuevas variables...
--- RESULTADOS ---
Precisión General (Accuracy): 0.7677
              precision    recall  f1-score   support

           0       0.86      0.85      0.86   1052339
           1       0.39      0.42      0.41    245474

    accuracy                           0.77   1297813
   macro avg       0.63      0.63      0.63   1297813
weighted avg       0.77      0.77      0.77   1297813

Matriz de Confusión:
Puntuales acertados: 893034 | Falsas Alarmas: 159305
Retrasos perdidos: 142165 | Retrasos detectados: 103309
Generando gráfico de importancia de variables...


C:\Users\Usuario\AppData\Local\Temp\ipykernel_2976\681480382.py:123: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=forest_importances.values, y=forest_importances.index, palette="viridis")


Analizando umbrales...
Mejor umbral: 0.68 (Acc: 0.8202)
📦 Comprimiendo ONNX...
✅ Todo listo en carpeta: artifacts
🔻 ZIP final: 13.21 MB


In [5]:
import joblib
import onnxruntime as rt
import numpy as np
import os

# --- CONFIGURACIÓN ---
ARTIFACTS_DIR = 'artifacts'

def cargar_cerebro():
    print("🧠 Cargando artefactos...")
    try:
        # Cargar los mapas (Diccionarios)
        maps = {}
        for col in ['CARRIER_NAME', 'DEPARTING_AIRPORT', 'DEP_TIME_BLK', 'PREVIOUS_AIRPORT']:
            maps[col] = joblib.load(f'{ARTIFACTS_DIR}/{col}_risk_map.joblib')
        
        # Cargar la media global (Red de seguridad)
        global_mean = joblib.load(f'{ARTIFACTS_DIR}/global_mean.joblib')
        
        # Cargar el modelo ONNX
        sess = rt.InferenceSession(f"{ARTIFACTS_DIR}/flight_delay_rf_weighted.onnx")
        
        return sess, maps, global_mean
    except Exception as e:
        print(f"❌ Error fatal: {e}")
        return None, None, None

def predecir_vuelo(json_input, sess, maps, global_mean):
    # 1. TRADUCCIÓN (El paso clave)
    # Convertimos texto (ej: "AA") a riesgo numérico (ej: 0.23)
    
    carrier_risk = maps['CARRIER_NAME'].get(json_input['CARRIER_NAME'], global_mean)
    airport_risk = maps['DEPARTING_AIRPORT'].get(json_input['DEPARTING_AIRPORT'], global_mean)
    time_risk = maps['DEP_TIME_BLK'].get(json_input['DEP_TIME_BLK'], global_mean)

    prev_airport = json_input.get('PREVIOUS_AIRPORT', 'UNKNOWN')
    prev_airport_risk = maps['PREVIOUS_AIRPORT'].get(prev_airport, global_mean)

    # 2. VECTOR DE ENTRADA (Orden ESTRICTO igual al entrenamiento)
    features = [
        json_input['MONTH'],
        json_input['DAY_OF_WEEK'],
        json_input['DISTANCE_GROUP'],
        json_input['SEGMENT_NUMBER'], 
        json_input['CONCURRENT_FLIGHTS'],
        json_input['PRCP'],
        json_input['TMAX'],
        json_input['AWND'],
        json_input['PLANE_AGE'],
        json_input['AIRPORT_FLIGHTS_MONTH'],
        
        # Riesgos calculados arriba
        carrier_risk,           
        airport_risk,           
        time_risk,
        
        # --- NUEVAS VARIABLES DE NEGOCIO (Faltaban estas 6) ---
        json_input.get('SNOW', 0.0),                  # Por defecto 0 si no se envía
        json_input.get('SNWD', 0.0),                  # Por defecto 0 si no se envía
        json_input.get('NUMBER_OF_SEATS', 150),       # Valor medio por defecto
        json_input.get('FLT_ATTENDANTS_PER_PASS', 0.005), # Valor medio por defecto
        json_input.get('GROUND_SERV_PER_PASS', 0.002),    # Valor medio por defecto
        prev_airport_risk                             # Riesgo calculado arriba
    ]

    # 3. PREDICCIÓN
    input_name = sess.get_inputs()[0].name
    # Convertir a float32 (ONNX es muy estricto con esto)
    input_data = np.array([features], dtype=np.float32)
    
    pred_onx = sess.run(None, {input_name: input_data})
    
    # Extraer probabilidad de retraso (Clase 1)
    # Random Forest en ONNX devuelve: [Etiqueta, Lista de Mapas de Probabilidad]
    probs = pred_onx[1][0] 
    prob_retraso = probs.get(1, 0.0)
    
    return prob_retraso

# --- ZONA DE PRUEBAS ---
if __name__ == "__main__":
    sess, maps, global_mean = cargar_cerebro()
    
    if sess:
        # CASO 1: Vuelo "Perfecto" (Sin lluvia, aerolínea buena)
        vuelo_bueno = {
            'MONTH': 5, 'DAY_OF_WEEK': 2, 'DISTANCE_GROUP': 3, 'SEGMENT_NUMBER': 1,
            'CONCURRENT_FLIGHTS': 10, 'PRCP': 0.0, 'TMAX': 25.0, 'AWND': 5.0,
            'PLANE_AGE': 3, 'AIRPORT_FLIGHTS_MONTH': 1000,
            'CARRIER_NAME': 'DL', 'DEPARTING_AIRPORT': 'ATL', 'DEP_TIME_BLK': '0600-0659'
        }

        # CASO 2: Vuelo "Pesadilla" (Lluvia, hora pico, aeropuerto congestionado)
        vuelo_malo = {
            'MONTH': 12, 'DAY_OF_WEEK': 5, 'DISTANCE_GROUP': 5, 'SEGMENT_NUMBER': 6,
            'CONCURRENT_FLIGHTS': 80, 'PRCP': 1.5, 'TMAX': 5.0, 'AWND': 25.0,
            'PLANE_AGE': 20, 'AIRPORT_FLIGHTS_MONTH': 5000,
            'CARRIER_NAME': 'AA', 'DEPARTING_AIRPORT': 'JFK', 'DEP_TIME_BLK': '1800-1859'
        }
        
        # CASO 3: Vuelo Internacional Desconocido (Tu pregunta sobre España/Brasil)
        vuelo_desconocido = {
            'MONTH': 7, 'DAY_OF_WEEK': 6, 'DISTANCE_GROUP': 10, 'SEGMENT_NUMBER': 1,
            'CONCURRENT_FLIGHTS': 20, 'PRCP': 0.0, 'TMAX': 30.0, 'AWND': 5.0,
            'PLANE_AGE': 5, 'AIRPORT_FLIGHTS_MONTH': 2000,
            'CARRIER_NAME': 'IBERIA',     # NO EXISTE EN EL DATASET
            'DEPARTING_AIRPORT': 'MAD',   # NO EXISTE EN EL DATASET (Madrid)
            'DEP_TIME_BLK': '1000-1059'
        }

        print("\n--- RESULTADOS DE LA PRUEBA ---")
        
        p1 = predecir_vuelo(vuelo_bueno, sess, maps, global_mean)
        print(f"✈️ Vuelo Bueno: {p1:.2%} de probabilidad de retraso.")

        p2 = predecir_vuelo(vuelo_malo, sess, maps, global_mean)
        print(f"⛈️ Vuelo Pesadilla: {p2:.2%} de probabilidad de retraso.")
        
        p3 = predecir_vuelo(vuelo_desconocido, sess, maps, global_mean)
        print(f"🌍 Vuelo Madrid->Brasil: {p3:.2%} (Usando media global por ser desconocido).")

🧠 Cargando artefactos...

--- RESULTADOS DE LA PRUEBA ---
✈️ Vuelo Bueno: 28.35% de probabilidad de retraso.
⛈️ Vuelo Pesadilla: 63.00% de probabilidad de retraso.
🌍 Vuelo Madrid->Brasil: 33.60% (Usando media global por ser desconocido).
